# Lesson 4: The 15-Minute City

**Planning question: Who can walk to a park or a school within 15 minutes?**

The **15-minute city** is a planning idea where daily needs such as parks, schools, shops and clinics are within a short walk or cycle of home.
In this notebook you will measure walking access in **Koramangala**, Bengaluru, using the real street network rather than straight lines.

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q osmnx

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

## Settings

An average adult walks at about 4.5 km per hour, which is 75 metres per minute.

In [ ]:
CENTER = (12.9352, 77.6245)   # Koramangala (latitude, longitude)
DIST = 2500                   # metres around the centre
WALK_SPEED = 75               # metres per minute

## Step 1: Download the walking network

We **project** the network to metres straight away, because we need lengths and distances.

In [ ]:
G = ox.graph_from_point(CENTER, dist=DIST, network_type="walk")
G = ox.project_graph(G)
graph_crs = G.graph["crs"]
print("Intersections:", len(G.nodes), " Street segments:", len(G.edges))

## Step 2: Add walking time to every street

Each street segment has a `length` in metres. Dividing by walking speed gives the minutes needed to walk it.

In [ ]:
for u, v, k, data in G.edges(keys=True, data=True):
    data["walk_min"] = data["length"] / WALK_SPEED

## Step 3: An isochrone from one point

An **isochrone** shows everywhere you can reach within a set time. We find the street node nearest to the centre,
then collect every street that is reachable within 5, 10 and 15 minutes.

In [ ]:
center_point = gpd.GeoSeries(gpd.points_from_xy([CENTER[1]], [CENTER[0]]), crs="EPSG:4326").to_crs(graph_crs)
center_node = ox.distance.nearest_nodes(G, X=center_point.x.iloc[0], Y=center_point.y.iloc[0])

isochrones = []
for minutes in [15, 10, 5]:
    reachable = nx.ego_graph(G, center_node, radius=minutes, distance="walk_min")
    reachable_streets = ox.graph_to_gdfs(reachable, nodes=False)
    shape = reachable_streets.buffer(40).union_all()
    isochrones.append({"minutes": minutes, "geometry": shape})

isochrones = gpd.GeoDataFrame(isochrones, crs=graph_crs)
isochrones

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)
fig, ax = plt.subplots(figsize=(9, 9))
edges.plot(ax=ax, color="lightgrey", linewidth=0.4)
isochrones.plot(ax=ax, column="minutes", cmap="viridis_r", alpha=0.6, legend=True,
                categorical=True, legend_kwds={"title": "Minutes walk"})
center_point.plot(ax=ax, color="red", markersize=60)
ax.set_title("Walking isochrones from the centre of Koramangala")
ax.set_axis_off()
plt.show()

Notice that the isochrones are not circles. Long blocks, railway lines and gated campuses all make the walking area smaller.

## Step 4: Download parks and schools

In [ ]:
parks = ox.features_from_point(CENTER, tags={"leisure": "park"}, dist=DIST)
parks = parks[parks.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(graph_crs)

schools = ox.features_from_point(CENTER, tags={"amenity": "school"}, dist=DIST).to_crs(graph_crs)

print("Parks:", len(parks), " Schools:", len(schools))

## Step 5: Walking time from every intersection to the nearest park

Here we treat every street intersection as a place where people live, which is a simple stand-in for homes.
We snap each park to its nearest street node, then calculate the shortest walk from **all** parks to **all** nodes in one step.

In [ ]:
def minutes_to_nearest(destinations):
    """Walking minutes from every node to the nearest destination."""
    points = destinations.centroid
    dest_nodes = ox.distance.nearest_nodes(G, X=points.x, Y=points.y)
    return nx.multi_source_dijkstra_path_length(G, sources=set(dest_nodes), weight="walk_min")

nodes["min_to_park"] = nodes.index.map(minutes_to_nearest(parks))
nodes["min_to_school"] = nodes.index.map(minutes_to_nearest(schools))
nodes[["min_to_park", "min_to_school"]].describe().round(1)

## Step 6: Map the access gaps

We group walking times into bands. The red areas are more than 15 minutes from a park. These are the **access gaps**.

In [ ]:
bins = [0, 5, 10, 15, 1000]
labels = ["0-5 min", "5-10 min", "10-15 min", "over 15 min"]
colors = {"0-5 min": "#1a9850", "5-10 min": "#91cf60", "10-15 min": "#fee08b", "over 15 min": "#d73027"}

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, column, title in zip(axes, ["min_to_park", "min_to_school"], ["Park", "School"]):
    nodes["band"] = pd.cut(nodes[column], bins=bins, labels=labels)
    edges.plot(ax=ax, color="lightgrey", linewidth=0.3)
    for label in labels:
        nodes[nodes["band"] == label].plot(ax=ax, color=colors[label], markersize=3, label=label)
    ax.set_title(f"Walking time to nearest {title.lower()}", fontsize=14)
    ax.legend(loc="lower left", markerscale=4)
    ax.set_axis_off()
plt.show()

## Step 7: The 15-minute score

What share of intersections are within a 15-minute walk of each service?

In [ ]:
summary = pd.DataFrame({
    "Within 5 min (%)": [(nodes[c] <= 5).mean() * 100 for c in ["min_to_park", "min_to_school"]],
    "Within 15 min (%)": [(nodes[c] <= 15).mean() * 100 for c in ["min_to_park", "min_to_school"]],
}, index=["Park", "School"]).round(1)
summary

## What this means for planning

- Network-based access is more honest than drawing a circle around a park, because it follows the routes people can actually walk.
- The red gaps are candidate locations for new parks, pocket parks or school sites.
- New pedestrian links, such as a gate in a long compound wall, can shrink the gaps at very low cost.

**Limitations:** intersections stand in for homes and do not reflect population. Parks just outside the study area are left out,
so access near the edges may look worse than it is. Park entrances are approximated by the park's centre point.

## Try it yourself

1. Repeat the analysis for clinics with `tags={"amenity": ["clinic", "hospital"]}`.
2. Change `WALK_SPEED` to 50 metres per minute for elderly people. How do the gaps grow?
3. Move `CENTER` to an outer area such as Electronic City `(12.8452, 77.6602)` and compare the scores.